In [ ]:
import os
import pandas as pd

# REMEBER TO CHANGE THE PATHS
pipe = pd.read_csv("/content/sales_pipeline.csv", parse_dates=["engage_date", "close_date"])
products = pd.read_csv("/content/products.csv")
teams = pd.read_csv("/content/sales_teams.csv")

# deals say "GTXPro", products file says "GTX Pro": make them match
products["product"] = products["product"].replace("GTX Pro", "GTXPro")

# same reference date as the spreadsheet's age_days formula
as_of = pd.Timestamp("2017-12-31")
print("Reference date:", as_of.date(), "| latest close date in data:", pipe["close_date"].max().date())

# ---- cutoffs from deals that already closed ----
closed = pipe[pipe["deal_stage"].isin(["Won", "Lost"])].copy()
closed["cycle_days"] = (closed["close_date"] - closed["engage_date"]).dt.days

won_cycle = closed.loc[closed["deal_stage"] == "Won", "cycle_days"]
check_in = int(won_cycle.quantile(0.75))
qualify_or_close = int(closed["cycle_days"].max())
print("Check-in threshold:", check_in, "| Qualify-or-close threshold:", qualify_or_close)

# ---- open deals: Engaging and Prospecting ----
open_deals = pipe[pipe["deal_stage"].isin(["Engaging", "Prospecting"])].copy()
open_deals["age_days"] = (as_of - open_deals["engage_date"]).dt.days   # NaN when engage_date is blank

open_deals = (open_deals.merge(products[["product", "sales_price"]], on="product", how="left")
                        .merge(teams, on="sales_agent", how="left"))

print("\nOpen deals by stage:")
print(open_deals["deal_stage"].value_counts())
print("Open deals with no engage_date:", int(open_deals["engage_date"].isna().sum()))
print("Open deals with no matching price:", int(open_deals["sales_price"].isna().sum()))

def tier(age):
    if pd.isna(age):           return "Not engaged yet"   # no engage_date, so no age to judge
    if age > qualify_or_close: return "Qualify or close"
    if age > check_in:         return "Check in"
    return "On track"

open_deals["tier"] = open_deals["age_days"].apply(tier)

tier_order = ["On track", "Check in", "Qualify or close", "Not engaged yet"]
print("\nTier breakdown:")
print(open_deals.groupby("tier").agg(deals=("opportunity_id", "count"),
      value=("sales_price", "sum")).reindex(tier_order).round(0))

# ---- stale share, among deals that have an age (Prospecting deals can't be judged) ----
engaged = open_deals[open_deals["age_days"].notna()]
stale_e = engaged["tier"] == "Qualify or close"

rep = engaged.groupby("sales_agent").agg(
    engaged_deals=("opportunity_id", "count"),
    stale=("tier", lambda s: (s == "Qualify or close").sum()),
    stale_value=("sales_price", lambda s: s[stale_e.loc[s.index]].sum()))
rep["stale_share"] = (rep["stale"] / rep["engaged_deals"]).round(2)
print("\nBy rep (highest stale share, engaged deals only):")
print(rep.sort_values("stale_share", ascending=False).head(10))

print("\nBy manager (engaged deals only):")
print(engaged.groupby("manager").agg(engaged_deals=("opportunity_id", "count"),
      stale_share=("tier", lambda s: round((s == "Qualify or close").mean(), 2))))

# ---- priority list: highest-value stale deals ----
priority = (open_deals[stale]
            .sort_values("sales_price", ascending=False)
            .head(15)[["opportunity_id", "sales_agent", "account", "product", "age_days", "sales_price"]])
print("\nTop 15 stale deals by value:")
print(priority)

# ---- save output ----
os.makedirs("output", exist_ok=True)
open_deals.to_csv("output/open_deals_tiered.csv", index=False)

Reference date: 2017-12-31 | latest close date in data: 2017-12-31
Check-in threshold: 88 | Qualify-or-close threshold: 138

Open deals by stage:
deal_stage
Engaging       1589
Prospecting     500
Name: count, dtype: int64
Open deals with no engage_date: 500
Open deals with no matching price: 0

Tier breakdown:
                  deals    value
tier                            
On track            108   221619
Check in            190   471946
Qualify or close   1291  3198664
Not engaged yet     500  1073986

By rep (highest stale share, engaged deals only):
                    engaged_deals  stale  stale_value  stale_share
sales_agent                                                       
Gladys Colclough               36     36        62513         1.00
Darcel Schlecht                83     83       273766         1.00
Cecily Lampkin                 19     19        29565         1.00
Marty Freudenburg              33     33       108803         1.00
Moses Frase                    34   